# Jak odhadnout x0 (počáteční odhad kořene)

Před použitím Newton / Steffensen / Halley / Fixed Point potřebuješ vědět,
**kde přibližně leží kořen**. Tento notebook ukazuje 4 způsoby jak na to.

---

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

print('Importy OK')

---
## Metoda 1 — Graf funkce

**Nejjednodušší.** Vykreslíš f(x) a vizuálně zjistíš kde kříží osu x.

**Příklad:** `f(x) = x³ - x - 2`

> **Pozor:** Rozsah  vždy přizpůsob funkci — musí pokrýt celou oblast kde čekáš kořeny. Začni širší, pak zužuj.


In [ ]:
f = lambda x: x**3 - x - 2

x_vals = np.linspace(-2, 3, 500)  # UPRAV rozsah dle funkce — musí pokrýt všechny kořeny!
y_vals = [f(x) for x in x_vals]

plt.figure(figsize=(8, 4))
plt.plot(x_vals, y_vals, label='f(x) = x³ - x - 2')
plt.axhline(0, color='red', linewidth=0.8, linestyle='--', label='y = 0')
plt.axvline(0, color='gray', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Kde f(x) kříží osu x → tam je kořen → to je tvůj x0')
plt.ylabel('f(x)')
plt.xlabel('x')
plt.show()

# Z grafu vidíš: kořen je někde kolem x ≈ 1.5  → x0 = 1.5

---
## Metoda 2 — Tabulka hodnot (změna znaménka)

Dosazuješ celá čísla a hledáš kde **f mění znaménko** → kořen leží mezi těmi dvěma hodnotami.

**Příklad:** `f(x) = e^x - 3x`

In [11]:
f = lambda x: math.exp(x) - 3*x

print(f"{'x':>6}  {'f(x)':>12}  znaménko")
print('-' * 32)

prev_sign = None
for xi in [i * 0.5 for i in range(-4, 10)]:
    fx = f(xi)
    sign = '+' if fx > 0 else '-'
    zmena = '  ← ZMĚNA ZNAMÉNKA → kořen zde!' if (prev_sign and sign != prev_sign) else ''
    print(f"{xi:>6.1f}  {fx:>12.4f}  {sign}{zmena}")
    prev_sign = sign

     x          f(x)  znaménko
--------------------------------
  -2.0        6.1353  +
  -1.5        4.7231  +
  -1.0        3.3679  +
  -0.5        2.1065  +
   0.0        1.0000  +
   0.5        0.1487  +
   1.0       -0.2817  -  ← ZMĚNA ZNAMÉNKA → kořen zde!
   1.5       -0.0183  -
   2.0        1.3891  +  ← ZMĚNA ZNAMÉNKA → kořen zde!
   2.5        4.6825  +
   3.0       11.0855  +
   3.5       22.6155  +
   4.0       42.5982  +
   4.5       76.5171  +


---
## Metoda 3 — Zdravý rozum z rovnice

Některé rovnice prozradí odhad přímo ze své struktury — bez počítání.

| Rovnice f(x) = 0 | Přepsání | Odhad x0 |
|-----------------|----------|----------|
| `x² - 5 = 0` | `x = √5` | `x0 = 2` (protože √5 ≈ 2.24) |
| `x³ - 2 = 0` | `x = ∛2` | `x0 = 1.5` (protože ∛2 ≈ 1.26) |
| `cos(x) - x = 0` | `cos(x) = x` | `x0 = 0.7` (průsečík cos a přímky y=x) |
| `e^x - 3x = 0` | `e^x = 3x` | viz tabulka/graf |
| `x³ - x - 1 = 0` | polynom stupně 3 | `x0 = 1` (zkus celá čísla) |

In [ ]:
# Rychlý test: dosadíš celá čísla a hledáš kde f(x) ≈ 0
f = lambda x: x**3 - x - 1

for xi in range(-3, 4):
    print(f'f({xi}) = {f(xi):.2f}')

# f(1) = -1, f(2) = 5 → kořen mezi 1 a 2 → x0 = 1.5

---
## Metoda 4 — Automatické hledání intervalů se změnou znaménka

Funkce projde rozsah a vrátí všechny intervaly kde f mění znaménko.
Použitelné když nevíš vůbec nic o funkci.

In [ ]:
def najdi_intervaly(f, a, b, krok=0.1):
    """
    Projde [a, b] s daným krokem a vrátí intervaly kde f mění znaménko.
    Každý interval obsahuje (přibližně) jeden kořen.
    """
    intervaly = []
    x = a
    while x + krok <= b:
        fa = f(x)
        fb = f(x + krok)
        if fa * fb < 0:  # změna znaménka
            stred = x + krok / 2
            intervaly.append((round(x, 6), round(x + krok, 6), round(stred, 6)))
        x += krok
    return intervaly


# Příklad: polynom se třemi kořeny (1, 2, 3)
f = lambda x: (x - 1) * (x - 2) * (x - 3)

intervaly = najdi_intervaly(f, -1, 5, krok=0.1)

print('Nalezené intervaly se změnou znaménka:')
for a, b, stred in intervaly:
    print(f'  [{a}, {b}]  →  x0 = {stred}  (f(x0) = {f(stred):.4f})')

In [ ]:
# Vizualizace všech nalezených kořenů
x_vals = np.linspace(-1, 5, 500)
y_vals = [f(x) for x in x_vals]

plt.figure(figsize=(8, 4))
plt.plot(x_vals, y_vals, label='f(x) = (x-1)(x-2)(x-3)')
plt.axhline(0, color='red', linewidth=0.8, linestyle='--')

for a, b, stred in intervaly:
    plt.axvspan(a, b, alpha=0.2, color='green')
    plt.plot(stred, f(stred), 'go', markersize=8, label=f'x0 ≈ {stred}')

plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Zelená oblast = nalezený interval, bod = doporučený x0')
plt.show()

---
## Shrnutí — kdy co použít

| Situace | Metoda |
|---------|--------|
| Máš čas, chceš to pochopit | Graf (metoda 1) |
| Rychlý odhad ručně | Tabulka celých čísel (metoda 2/3) |
| Nevíš vůbec nic o funkci | `najdi_intervaly()` (metoda 4) |
| Máš interval [a,b] | x0 = střed = (a+b)/2 |

**Klíčové pravidlo:**
> Stačí najít dvě čísla kde f má **opačná znaménka**. Kořen leží mezi nimi. Střed = dobrý x0.

Pro **bisekci a regula falsi** nepotřebuješ x0 — stačí interval [a, b] kde f(a)·f(b) < 0.